In [13]:
import numpy as np
import pandas as pd

## UMDA One Max

In [14]:

def create_pop_binaria_umdaom(numero_bits, tamaño_poblacion):
    return np.random.randint(0, 2, (tamaño_poblacion, numero_bits))

def onemax_umdaom(x):
    return np.sum(x)

def umda_one_max(config):
    numero_bits = config["Numero de bits"]
    n = config["Tamaño de la poblacion"]
    n_mejores = max(2, int(np.ceil(n * config["Fraccion a mantener"])))
    iteraciones = config["Numero de iteraciones"]
    seed = config["Semilla"]
    if seed is not None:
        np.random.seed(seed)
    pop = create_pop_binaria_umdaom(numero_bits, n)
    fitness = np.array([onemax_umdaom(ind) for ind in pop])
    for _ in range(iteraciones):
        idx = np.argsort(-fitness)
        mejores = pop[idx][:n_mejores]
        p = np.mean(mejores, axis=0)
        pop = (np.random.rand(n, numero_bits) < p).astype(int)
        fitness = np.array([onemax_umdaom(ind) for ind in pop])
    mejor_fitness = np.max(fitness)
    return mejor_fitness

## MIMIC One Max

In [ ]:

def create_pop_mimicom(pop_size, n_bits):
    return np.random.randint(0, 2, size=(pop_size, n_bits)).astype(np.int8)
def evaluate_pop_mimcom(pop):
    # Suma de bits por individuo (problema OneMax)
    return pop.sum(axis=1)
def estimate_marginals_and_conditionals_mimcom(selected):
    m, n = selected.shape 
    counts1 = selected.sum(axis=0).astype(np.float64)
    marginals = (counts1 + 1) / (m + 2)
    cond = np.zeros((n, n, 2), dtype=np.float64)
    for j in range(n):
        idx0 = np.where(selected[:, j] == 0)[0]
        idx1 = np.where(selected[:, j] == 1)[0]
        for i in range(n):
            c0 = selected[idx0, i].sum() if idx0.size > 0 else 0.0
            c1 = selected[idx1, i].sum() if idx1.size > 0 else 0.0
            n0 = idx0.size
            n1 = idx1.size
            p_i_given_j0 = (c0 + 1) / (n0 + 2) if n0 > 0 else marginals[i]
            p_i_given_j1 = (c1 + 1) / (n1 + 2) if n1 > 0 else marginals[i]
            cond[i, j, 0] = p_i_given_j0
            cond[i, j, 1] = p_i_given_j1
    return marginals, cond

def bernoulli_entropy_mimicom(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

def conditional_entropy_i_given_j_mimicom(i, j, marginals, cond):
    p_j = marginals[j]
    h0 = bernoulli_entropy_mimicom(cond[i, j, 0])
    h1 = bernoulli_entropy_mimicom(cond[i, j, 1])
    return (1 - p_j) * h0 + p_j * h1

def build_greedy_order_mimicom(n, marginals, cond):
    remaining = set(range(n))
    entropies = [bernoulli_entropy_mimicom(marginals[i]) for i in range(n)]
    first = int(np.argmin(entropies))
    order = [first]
    remaining.remove(first)
    while remaining:
        last = order[-1]
        best_j = None
        best_h = None
        for j in remaining:
            h = conditional_entropy_i_given_j_mimicom(j, last, marginals, cond)
            if best_j is None or h < best_h:
                best_j = j
                best_h = h
        order.append(best_j)
        remaining.remove(best_j)
    return order

def sample_from_chain_mimicom(order, marginals, cond, n_samples):
    n = len(order)
    samples = np.zeros((n_samples, n), dtype=np.int8)
    for s in range(n_samples):
        first_idx = order[0]
        prob1 = marginals[first_idx]
        x_first = 1 if np.random.rand() < prob1 else 0
        samples[s, first_idx] = x_first
        prev_idx = first_idx
        prev_val = x_first
        for pos in range(1, n):
            idx = order[pos]
            p = cond[idx, prev_idx, prev_val]
            val = 1 if np.random.rand() < p else 0
            samples[s, idx] = val
            prev_idx = idx
            prev_val = val
    return samples

def mimic_one_max(config):
    n_bits = config["Numero de bits"]
    pop_size = config["Tamaño de la poblacion"]
    frac = config["Fraccion a mantener"]
    max_iters = config["Numero de iteraciones"]
    seed = config["Semilla"]
    
    if seed is not None:
        np.random.seed(seed)
    
    pop = create_pop_mimicom(pop_size, n_bits)
    fitness_pop = evaluate_pop_mimcom(pop)
    best_global_fit = -1
    
    for _ in range(max_iters):
        keep_k = max(2, int(np.ceil(frac * pop_size)))
        idx_sorted = np.argsort(-fitness_pop)
        selected = pop[idx_sorted[:keep_k]]
        selected_fits = fitness_pop[idx_sorted[:keep_k]]
        if selected_fits[0] > best_global_fit:
            best_global_fit = int(selected_fits[0])

        marginals, cond = estimate_marginals_and_conditionals_mimcom(selected)
        order = build_greedy_order_mimicom(n_bits, marginals, cond)
        pop = sample_from_chain_mimicom(order, marginals, cond, pop_size)
        fitness_pop = evaluate_pop_mimcom(pop)

    return best_global_fit


## MIMIC Random One Max

In [16]:
import numpy as np

def create_pop_mimicrom(pop_size, n_bits):
    return np.random.randint(0, 2, size=(pop_size, n_bits)).astype(np.int8)

def evaluate_pop_mimicrom(pop):
    return pop.sum(axis=1)

def estimate_marginals_and_conditionals_mimicrom(selected):
    m, n = selected.shape
    counts1 = selected.sum(axis=0).astype(np.float64)
    marginals = (counts1 + 1) / (m + 2)

    cond = np.zeros((n, n, 2), dtype=np.float64)
    for j in range(n):
        idx0 = np.where(selected[:, j] == 0)[0]
        idx1 = np.where(selected[:, j] == 1)[0]
        n0, n1 = idx0.size, idx1.size
        for i in range(n):
            c0 = selected[idx0, i].sum() if n0 > 0 else 0.0
            c1 = selected[idx1, i].sum() if n1 > 0 else 0.0
            p_i_given_j0 = (c0 + 1) / (n0 + 2) if n0 > 0 else marginals[i]
            p_i_given_j1 = (c1 + 1) / (n1 + 2) if n1 > 0 else marginals[i]
            cond[i, j, 0] = p_i_given_j0
            cond[i, j, 1] = p_i_given_j1
    return marginals, cond

def sample_from_chain_random_order_mimicrom(order, marginals, cond, n_samples):
    n = len(order)
    samples = np.zeros((n_samples, n), dtype=np.int8)
    for s in range(n_samples):
        first_idx = order[0]
        p_first = marginals[first_idx]
        x_first = 1 if np.random.rand() < p_first else 0
        samples[s, first_idx] = x_first
        prev_idx = first_idx
        prev_val = x_first
        for pos in range(1, n):
            idx = order[pos]
            p = cond[idx, prev_idx, prev_val]
            val = 1 if np.random.rand() < p else 0
            samples[s, idx] = val
            prev_idx = idx
            prev_val = val
    return samples

def mimic_aleatorio_one_max(config):
    n_bits = config["Numero de bits"]
    pop_size = config["Tamaño de la poblacion"]
    frac = config["Fraccion a mantener"]
    max_iters = config["Numero de iteraciones"]
    seed = config["Semilla"]

    if seed is not None:
        np.random.seed(seed)

    pop = create_pop_mimicrom(pop_size, n_bits)
    fitness_pop = evaluate_pop_mimicrom(pop)
    best_global_fit = int(fitness_pop.max())

    for _ in range(max_iters):
        keep_k = max(2, int(np.ceil(frac * pop_size)))
        idx_sorted = np.argsort(-fitness_pop)
        selected = pop[idx_sorted[:keep_k]]
        selected_fits = fitness_pop[idx_sorted[:keep_k]]

        if selected_fits[0] > best_global_fit:
            best_global_fit = int(selected_fits[0])

        marginals, cond = estimate_marginals_and_conditionals_mimicrom(selected)

        order = np.random.permutation(n_bits)

        pop = sample_from_chain_random_order_mimicrom(order, marginals, cond, pop_size)
        fitness_pop = evaluate_pop_mimicrom(pop)
        if best_global_fit == n_bits:
            break
    return best_global_fit




## Definiendo parametros para Expereminetacion One Max para cada algoritmo

In [17]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed

# --- Es necesario que instales joblib si no lo tienes ---
# pip install joblib

# --- Supongamos que tus funciones están definidas aquí ---
# def umda_one_max(config): ...
# def mimic_one_max(config): ...
# def mimic_aleatorio_one_max(config): ...
# (Asegúrate de que estas funciones devuelven el 'mejor_fitness')

def run_experiment():
    """Función que contiene toda la lógica del experimento."""
    
    n_bits_pruebas = [20, 50, 75, 100]
    num_ejecuciones = 33

    algoritmos_config = {
        "UMDA onemax": {
            "funcion": umda_one_max,
            "tam_poblacion": [100, 250, 400, 600],
            "fraccion_mantener": [0.5, 0.5, 0.5, 0.5],
            "num_iteraciones": [50, 100, 150, 200]
        },
        "MIMIC onemax": {
            "funcion": mimic_one_max,
            "tam_poblacion": [200, 600, 1000, 1500],
            "fraccion_mantener": [0.3, 0.25, 0.2, 0.2],
            "num_iteraciones": [60, 120, 180, 250]
        },
        "MIMIC aleatorio onemax": {
            "funcion": mimic_aleatorio_one_max,
            "tam_poblacion": [200, 600, 1000, 1500],
            "fraccion_mantener": [0.3, 0.25, 0.2, 0.2],
            "num_iteraciones": [60, 120, 180, 250]
        }
    }

    resultados_totales_onemax = []

    for nombre_alg, config_alg in algoritmos_config.items():
        funcion_alg = config_alg["funcion"]
        
        for idx, n_bits in enumerate(n_bits_pruebas):
            pop_size = config_alg["tam_poblacion"][idx]
            frac = config_alg["fraccion_mantener"][idx]
            max_iters = config_alg["num_iteraciones"][idx]
            
            print(f"--- Preparando ejecuciones paralelas para {nombre_alg} con {n_bits} bits... ---")

            # Preparamos una lista de todas las configuraciones para las 33 ejecuciones
            configs_para_paralelo = []
            for semilla in range(num_ejecuciones):
                configs_para_paralelo.append({
                    "Numero de bits": n_bits,
                    "Tamaño de la poblacion": pop_size,
                    "Fraccion a mantener": frac,
                    "Numero de iteraciones": max_iters,
                    "Semilla": None 
                })

            # ▼▼▼ ESTA ES LA PARTE PARALELIZADA ▼▼▼
            # Ejecuta las 33 llamadas a 'funcion_alg' en paralelo en todos los núcleos de CPU disponibles.
            # 'verbose=10' muestra una barra de progreso.
            fitness_runs = Parallel(n_jobs=-1, verbose=10)(
                delayed(funcion_alg)(config) for config in configs_para_paralelo
            )
            # ▲▲▲ FIN DE LA PARTE PARALELIZADA ▲▲▲
            
            # Estadísticas (el resto del código es igual)
            media = np.mean(fitness_runs)
            mediana = np.median(fitness_runs)
            desviacion = np.std(fitness_runs)
            
            resultados_totales_onemax.append({
                "Algoritmo": nombre_alg,
                "Bits": n_bits,
                "Poblacion": pop_size,
                "Fraccion": frac,
                "Iteraciones": max_iters,
                "Media": media,
                "Mediana": mediana,
                "Desviacion": desviacion
            })

    df_resultados = pd.DataFrame(resultados_totales_onemax)
    df_resultados = df_resultados.sort_values(by=["Algoritmo", "Bits"]).reset_index(drop=True)

    print("\nResumen de resultados:")
    print(df_resultados.to_markdown(index=False))
    return df_resultados



In [18]:

from scipy.stats import friedmanchisquare
import scikit_posthocs as sp

def analizar_estadisticamente(df_resultados):
    """
    Aplica el test de Friedman sobre las medias del dataframe y,
    si el resultado es significativo, realiza un post-hoc Nemenyi.
    """
    # --- Reestructurar datos: filas = tamaño de bits, columnas = algoritmos ---
    tabla_medias = df_resultados.pivot_table(
        index="Bits", columns="Algoritmo", values="Media"
    )

    print("\n=== Tabla de medias para análisis estadístico ===")
    print(tabla_medias.round(4))

    # --- Test de Friedman ---
    valores = [tabla_medias[col].values for col in tabla_medias.columns]
    stat, p = friedmanchisquare(*valores)

    print("\n=== Test de Friedman ===")
    print(f"Estadístico: {stat:.4f}")
    print(f"P-valor: {p:.6f}")

    # --- Evaluación del resultado ---
    alpha = 0.05
    if p < alpha:
        print(f"\n✅ Diferencias significativas detectadas (p < {alpha}).")
        print("→ Aplicando post-hoc de Nemenyi...\n")

        # --- Post hoc de Nemenyi ---
        p_posthoc = sp.posthoc_nemenyi_friedman(tabla_medias)
        print("=== Resultados post-hoc (p-values) ===")
        print(p_posthoc.round(4))

        print("\nInterpretación: valores p < 0.05 indican diferencias significativas entre esos algoritmos.")
    else:
        print(f"\nNo se detectan diferencias significativas (p ≥ {alpha}).")

# --- Uso dentro del flujo de tu experimento ---
if __name__ == "__main__":
    df_resultados = run_experiment()  # Asegúrate de que la función lo devuelva
    analizar_estadisticamente(df_resultados)


--- Preparando ejecuciones paralelas para UMDA onemax con 20 bits... ---
--- Preparando ejecuciones paralelas para UMDA onemax con 50 bits... ---


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.02382636070251465s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   6 out of  33 | elapsed:    0.0s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  10 out of  33 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  14 out of  33 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  18 out of  33 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  22 out of  33 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  26 out of  33 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  30 out of  33 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  33 out of  33 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1087181568145752s.) Setting batch_size=2.
[Parallel(n_jobs=

--- Preparando ejecuciones paralelas para UMDA onemax con 75 bits... ---


[Parallel(n_jobs=-1)]: Done   6 out of  33 | elapsed:    0.3s remaining:    1.5s
[Parallel(n_jobs=-1)]: Done  10 out of  33 | elapsed:    0.3s remaining:    0.8s
[Parallel(n_jobs=-1)]: Done  14 out of  33 | elapsed:    0.4s remaining:    0.5s
[Parallel(n_jobs=-1)]: Done  18 out of  33 | elapsed:    0.5s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done  22 out of  33 | elapsed:    0.5s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done  26 out of  33 | elapsed:    0.6s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  30 out of  33 | elapsed:    0.6s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  33 out of  33 | elapsed:    0.7s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.


--- Preparando ejecuciones paralelas para UMDA onemax con 100 bits... ---


...

[Parallel(n_jobs=-1)]: Done   6 out of  33 | elapsed:    1.1s remaining:    5.5s
[Parallel(n_jobs=-1)]: Done  10 out of  33 | elapsed:    1.2s remaining:    2.8s
[Parallel(n_jobs=-1)]: Done  14 out of  33 | elapsed:    1.6s remaining:    2.2s
[Parallel(n_jobs=-1)]: Done  18 out of  33 | elapsed:    2.2s remaining:    1.8s
[Parallel(n_jobs=-1)]: Done  22 out of  33 | elapsed:    2.3s remaining:    1.1s
[Parallel(n_jobs=-1)]: Done  26 out of  33 | elapsed:    2.4s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done  30 out of  33 | elapsed:    2.6s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done  33 out of  33 | elapsed:    2.7s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.


--- Preparando ejecuciones paralelas para MIMIC aleatorio onemax con 100 bits... ---


[Parallel(n_jobs=-1)]: Done   6 out of  33 | elapsed:    2.5s remaining:   11.8s
[Parallel(n_jobs=-1)]: Done  10 out of  33 | elapsed:    2.7s remaining:    6.2s
[Parallel(n_jobs=-1)]: Done  14 out of  33 | elapsed:    3.7s remaining:    5.1s
[Parallel(n_jobs=-1)]: Done  18 out of  33 | elapsed:    5.2s remaining:    4.3s
[Parallel(n_jobs=-1)]: Done  22 out of  33 | elapsed:    5.2s remaining:    2.6s
[Parallel(n_jobs=-1)]: Done  26 out of  33 | elapsed:    5.3s remaining:    1.4s
[Parallel(n_jobs=-1)]: Done  30 out of  33 | elapsed:    6.0s remaining:    0.5s



Resumen de resultados:
| Algoritmo              |   Bits |   Poblacion |   Fraccion |   Iteraciones |   Media |   Mediana |   Desviacion |
|:-----------------------|-------:|------------:|-----------:|--------------:|--------:|----------:|-------------:|
| MIMIC aleatorio onemax |     20 |         200 |       0.3  |            60 |      20 |        20 |            0 |
| MIMIC aleatorio onemax |     50 |         600 |       0.25 |           120 |      50 |        50 |            0 |
| MIMIC aleatorio onemax |     75 |        1000 |       0.2  |           180 |      75 |        75 |            0 |
| MIMIC aleatorio onemax |    100 |        1500 |       0.2  |           250 |     100 |       100 |            0 |
| MIMIC onemax           |     20 |         200 |       0.3  |            60 |      20 |        20 |            0 |
| MIMIC onemax           |     50 |         600 |       0.25 |           120 |      50 |        50 |            0 |
| MIMIC onemax           |     75 |        1000 

[Parallel(n_jobs=-1)]: Done  33 out of  33 | elapsed:    6.4s finished
c:\Users\chris\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\stats\_stats_py.py:8780: RuntimeWarning: invalid value encountered in scalar divide
  statistic = (12.0 / (k*n*(k+1)) * ssbn - 3*n*(k+1)) / c
